# Build Segment-Based Library

The notebooks use the Wildcat Creek as an example to introduce the FLDPLN model and to map the 2018 Labor Day flood inundation. [Wildcat Creek data](https://kbs-karsfl-pc01.home.ku.edu/download/wildcat_10m_3dep.zip) is available as a zip file on KU KBS-KARS server as it's too large to be put on Github. The zip file contains DEM and its derivatives, segment shapefiles, gauge Excel file and a reference flood inundation map.

In this notebook, we will first identify streams, divide them into segments and create a segment-based library for the Wildcat Creek.

## Create a FLDPLN Model Object

To use the FLDPLN model, we first need to create a FLDPLN model object which automatically initializes the model's Python package (i.e., fldpln_py). This will activate the MATLAB Runtime and take a little bit of time.

In [1]:
import os

# import FLDPLN model from fldpln_model package
from fldpln_model import  FLDPLN # or from fldpln_model.model import FLDPLN

# Create a FLDPLN model object
fldplnModel = FLDPLN()

Initialize FLDPLN model python package ...


## Generate Segments

The first step to create a FLDPLN library is to identify stream networks from a hydro-conditioned digital elevation model (DEM) and divide the stream networks into segments. The stream networks are first identified using a flow accumulation threshold (i.e., the strfac parameter). The stream networks are then divided into natural reaches (stream links in ArcGIS' term) which are streams between headwater and confluence pixels or between two consecutive confluence pixels. Note that pixels flow out of the DEM are treated as confluence pixels in this process. Those natural reaches are further divided into segments if the flow accumulation along its path jumps greater than or equal to another threshold (i.e., the segfac parameter). Then the segments are bisected until all segments are not longer than a certain length threshold (i.e., the seglen parameter). Note that the units for the flow accumulation thresholds are in squared miles and the unit for segment length is in miles.

The default values for the 3 parameters (i.e., strfac, segfac and seglen) are: 70 sq. miles, 25 sq. miles, and 2 miles for the libraries in KS. Note that the stream networks of Kansas FLDPLN libraries are more coarse grained than the National Water Model (NWM) reaches. This is because the KS stream networks are generated just to use available USGS gauges. If the NWM discharge is with FDLPLN libraries for the flood inundation mapping, the stream networks should to be densified.

In [2]:
# Use Wildcat creek as an example
bildir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/bil'
segdir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/segs'

# Input flow direction and accumulation BIL files
fdrf = os.path.join(bildir, 'fdr.bil')
facf = os.path.join(bildir, 'fac.bil')

# segment parameters
strfac = 50; # flow accumulation threshold (in sq. miles) for identifying stream networks. 50 for Wildcat to include the upstream gauge. 70 for Verdigris and most KS watersheds
segfac = 25; # flow accumulation threshold (in sq. miles) for segment stream networks. 25 is the default in KS
seglen = 2; # segment length in miles. usually is the SQRT of segfac. 2 for Wildcat and 5 for Verdigris and others in KS

# generate segments
fldplnModel.GenerateSegments(fdrf, facf, strfac, segfac, seglen, segdir)

# write FSP information as a MAT file for use in generating stream order and segment information as a CSV files
# This is necessary when generating stream/segment orders!
seg_list = []; # [] for all the segments or a list of selected segment IDs (i.e., seg_list = [1,2,3]) 
fldplnModel.WriteSegmentFspCsvFiles(bildir, segdir, seg_list, segdir, 'mat')

## Select Segments and Create Spatial Mask

We don't need to generate a FLDPLN library for all the segment extracted from the DEM. Typically we want to exclude segments within or in the vicinity of large waterbodies (such as reservoirs and lakes) as the inundation from those waterbodies are different from river inundation.

### Generate Segment Shapefile 

Here we use the ArcGIS Pro's Stream to Features tool with str_segid.bil and fdr.bil as the input rasters and **Simplify polyline unchecked**. The segment shapefiles can then be used to select a subset of segments for running the FLDPLN model.

Note that the Raster to Polyline tool in ArcGIS Pro doesn't work properly in some cases. In the figure below, the left stream polyline is generated by the Raster to Polyline tool, and the right stream polyline is generated by the Stream to Feature tool. The left polyline is erroneous. Also note that the Stream to Features tool connects upstream segments at the downstream confluence pixel.

![](./images/stream2feature.PNG)

### Select Segments

Using the shapefiles generated from the above step, we can exam the stream networks and segments and see whether they are appropriate, and may re-generate them if necessary.

With the segment shapefile, we can select the segments in a sub-watershed or any subset of segments to create a FLDPLN library. Take a note on the field (usually grid_code) that stores the segment ID in the shapefile. We can exported the selected subset of segments as a new shapefile, which can then be used to generate their FLDPLN library.

### Prepare Spatial Mask

Areas surrounding large waterbodies such as reservoirs and lakes should be masked out and NOT included before running the FLDPLN model as their flood inundation, which is different from rivers, is primarily caused by their water levels which are usually controlled by human or USACE in the United States. Based on Kansas experience, for most lakes/reservoirs, the inundation extent of their 75 percentile of water surface elevation is a good mask to use. All the segments that intersect with this mask should be removed from the segment shapefile for generating FLDPLN library.  

Note that the mask raster must be in the BIL format. When a mask BIL raster is provided as the input parameter when generating a FLDPLN  library, those cells within the mask, including both stream cells and floodplain cells, are remove from the modeling. 

## Create Segment-Based Library

With the segments decided (and possibly mask created), we can create a FLDPLN library for the selected segments. When using the selected segment shapefile, we need to specify the segment ID field/column in the shapefile. All the segments by default use the same maximum flood stage asa specified by the "fldmx" model parameter. We can also add a new field in the shapefile to use different maximum flood stages for the segments.   

**Note that this is the most time consuming step in building a FLDPLN libraries, especially when the maximum flood stage (i.e., fldmx)modelled is high.**

In [3]:
# Set folders
rawdir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/rawlib' # raw segment-based library
subsegShapefile = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/segment_shapefiles/wildcat_segments.shp' # subset of segment shapefile for Wildcat creek

# Applying a spatial mask if provided. The mask removes the waterbodies and potentially inundated areas by them from the FLDPLN model
filmskfile = '' # no spatial mask for Wildcat. Set to '' for no mask
# filmskfile = 'E:/fldpln/workshop/devcon26/verdigris_10m/bil/fil_masked.bil' # for Verdigris River

# Set a subset of segments in the shapefile
# The subset of segments and their modelled max stage are defined by a dictionary of 3 keys: 'file', 'segid_field', and 'seg_fldmx_field', 
# where 'file' is the shapefile, 'segid_field' is the field name for segment ID, and 'seg_fldmx_field' is the field name for the max stage.
segshpfile = {'file':subsegShapefile} # set to {'file': ''} to include all segments and use the same fldmx for all segments
# set the two additional keys/values if a segment shapefile is provided
segshpfile['segid_field'] = 'grid_code' # field name for segment ID
segshpfile['seg_fldmx_field'] = '' # set to '' when all the segments use the same fldmx. Otherwise specify a field in the shapefile

# Set FLDPLN model parameters
fldmn = 0.01 # typically set to 1 centimeter or 0.0328084 foot deoends on DEM's vertical unit
fldmx = 15 # max. stage modeled in DEM's vertical unit. Both wildcat_10m_3dep and verdigris DEM vertical unit is in meters
dh = 1 # vertical step size in DEM's vertical unit
mxht = 0 # max(dem+flood height) to cease flooding. Usually set to 0 for no capped height/elevation

# Set FLDPLN model running parameters
# model run memory type: choose one from {'hd', 'ram0', 'ram'}. 
mtype = 'ram' # choose either 'ram' (machine has RAM >= 64G) or 'hd' (uses least RAM)

# parallelization parameters can be defined by a dictionary of 3 keys: 'type', 'numcores', and 'worker_type', 
# where 'type' is the parallelization type, 'numcores' is the number of cores, and 'worker_type' is the type of worker.
# set the parallelization type. Choose one from {'none', 'parfor', 'parfeval'} but 'parfeval' is preferred
para = {'type': 'parfeval'}
# set parallelization parameter keys if the model is run in parallel (i.e., para['type'] is not 'none')
# set the number of cores, and if not set the max number of cores will be used
para['numcores'] = 2 # Comment out to use all the cores available
# set the worker type. Choose one from {'Threads', 'Processes'}. 'Threads' is not supported by compiled Python package!
para['worker_type'] = 'Processes'

# Run the model to create a segment-based library
fldplnModel.CreateSegmentLibrary(bildir, segdir, filmskfile, segshpfile, fldmn, fldmx, dh, mxht, rawdir, mtype, para)

## Generate Stage-Volume Tables

Use the library we can generate stage-volume relations at various stages and save them in a table for each segment. Those tables will be used when interpolating stream stages with observed gauge stages to map inundation. 

In [4]:
# Generate stage-volume table for the segments in raw segment-based library
rawdir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/rawlib' # segment-based library directory
svtdir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/src_stats' # output directory for the stage-volume tables

# Set parameters for generating stage-volume table
stage_step = 0.003048; # in meters or 0.01 in feet based on DEM's vertical unit for generating stage-volume table
cell_size = 10 # cell size of the DEM in the segment-based library in meters   

# Set a subset of segments and potentially apply adjustment factors to segments
# segshpfile = {'file':'E:/fldpln/matlab2py/examples/wildcat_10m_3dep/segment_shapefiles/wildcat_segments.shp'} # shapefile for the segments to be included in the stage-volume table.
segshpfile = {'file':''} # Set to '' to include all segments
segshpfile['segid_field'] = 'grid_code' # field name in the shapefile that contains the segment ID
segshpfile['seg_adjust_field'] = '' # field name in the shapefile that contains the adjustment factor for the stage-volume table. Set to '' if no adjustment is needed.

# Generate stage-volume table for the segments
fldplnModel.GenerateStageVolumeTable(rawdir, stage_step, cell_size, segshpfile, svtdir)

## Format Segment-Based Library for Tiling

The raw segment-based library generated in the previous step doesn't have map coordinates and is not efficient for mapping. Here we reformat the raw library and make it ready for tiling and efficient inundation mapping.

In [5]:
# Folder for reformatted library
seglibdir = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/seglib' # reformatted library for tiling and mapping

# Reformat raw segment library for tiling and mapping
fldplnModel.FormatSegmentLibrary(bildir, segdir, rawdir, seglibdir)

## Generate Stream/Segment Orders (optional)

Stream/segment orders are used while interpolating the stream depth of flow (DOF) from observed gauge stages using earlier interpolation methods (i.e., horizontal and vertical methods). Interpolation is performed from low order to high order streams/segments. The general rules for assigning stream order are:
* A stream (or more accurately a reach) consists of several connected segments which flows out of the DEM or into another stream at a confluence.
* Lower order streams are main streams where FSP DOF are first interpolated first and confluences on the main streams may serve as ggauges when interpolating in-flow high order tributary streams.
* Stream order must be unique in a library. No two streams can have the same order even if they are in different sub-stream networks in the library

The above requirements are necessary to make sure the interpolation is carried out correctly. Stream/segment orders can be assigned either manually using a GIS software or automatically using the code below which is a relatively new function still under testing and verification.

**With the newly developed volume-based stage interpolation method, we don't need to generate stream/segment orders anymore unless we want to use horizontal or vertical interpolation methods.**

In [6]:
# Generate stream order for the segments in Wildcat Creek.
segshp = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/segment_shapefiles/wildcat_segments.shp' 

# Four files are created in the segdir folder by the method: 
# network_slopes4stream_ordering.mat and network_slopes4stream_ordering.xlsx file
# Two shapefiles with subnetwwork ID and stream order within the subnetwork
# The "strord" field in the *_segments_ord4map shapefile has the stream order. Other files are for reference and debugging. 

fldplnModel.GenerateStreamOrder(bildir, segdir, segshp)

## Terminate the FLDPLN Model

This is done by delete the FLDPLN model object.

In [7]:
del fldplnModel # or fldplnModel = None

## Issues and Questions

* Messages displayed in the fldpln_model Python package are NOT shown in this notebook but running the code cells in a python script (xp_fldpln_model_wildcat.py) the messages are shown just fine.
